In [1]:
import os
from dotenv import load_dotenv

load_dotenv()  # 加载环境变量
API_KEY = os.getenv("API_KEY")
BASE_URL = os.getenv("BASE_URL")
if not API_KEY:
    raise ValueError("未检测到 API_KEY，请检查 .env 文件是否配置正确")

In [2]:
#(1)调用OpenAI的ChatModel
from langchain_openai import ChatOpenAI

#1.初始化对话模型
chat_model=ChatOpenAI(
    model="mimo-v2.5-pro", #选择对话模型
    api_key=API_KEY,
    base_url=BASE_URL,
    temperature=0.3,  # 随机性：0-1，越小越严谨，越大越有创造力
    max_tokens=200   # 最大生成 tokens 数，避免生成过长内容
)

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [3]:
#2. 构造对话模型
#ChatModel需要接收的是"消息列表",每个消息有角色(user/assistant/system)和内容
messages=[
    #system消息:给助手设定身份和行为准则，会影响后续所有回复
    {'role':"system","content":"你是一个耐心的AI学习助手,回复简洁易懂,适合高校学生理解。"},
    #user消息: 用户的问题
    {"role":"user","content":"请用3句话解释什么是LangChain?"}
]

In [4]:
#3. 调用模型生成结果
#统一调用方法: invoke(),传入消息列表
result=chat_model.invoke(messages)

In [5]:
#4.输出结果
#结果是一个ChatMessage对象,content属性是回复内容
print("ChatModel回复: ")
print(result.content)

ChatModel回复: 
LangChain 是一个用于开发由大型语言模型（LLM）驱动的应用程序的开源框架。  
它通过提供模块化组件和预置的链（Chains），帮助开发者便捷地连接 LLM、外部数据及工具。  
其主要应用于构建聊天机器人、问答系统、文档分析等智能应用。


In [6]:

# 初始化对话历史（包含 system 设定）
history = [
    {"role": "system", "content": "你是一个耐心的AI学习助手，回复简洁易懂，适合高校学生理解。"}
]

# 第一轮对话
history.append({"role": "user", "content": "请用3句话解释什么是LangChain？"})

result = chat_model.invoke(history)
print("【第一轮回复】：")
print(result.content)

# 将模型的回复添加到历史中（assistant 消息）
history.append({"role": "assistant", "content": result.content})

# 第二轮对话
# 追问，模型需要上下文才能理解"它"
history.append({"role": "user", "content": "它的核心组件有哪些？"})

result = chat_model.invoke(history)
print("\n【第二轮回复】：")
print(result.content)

# 继续记录
history.append({"role": "assistant", "content": result.content})

# 第三轮对话
history.append({"role": "user", "content": "给我一个简单的使用场景"})

result = chat_model.invoke(history)
print("\n【第三轮回复】：")
print(result.content)



【第一轮回复】：
LangChain是一个用于构建大语言模型应用的工具包，它可以帮助你将语言模型与外部数据、工具和服务连接起来。它简化了开发流程，让开发者能更高效地创建如聊天机器人、智能助手等应用。通过LangChain，你可以轻松实现文档问答、数据分析、任务自动化等功能。

【第二轮回复】：


【第三轮回复】：
**场景：智能课件问答助手**

假设你是一名大学生，课件有100页PDF。你可以用LangChain构建一个应用：

1. **加载课件** → 将PDF文档读入系统
2. **分割文本** → 把长文档拆成小段落
3. **创建向量索引** → 让系统"理解"内容含义
4. **提问** → 你问："期末考试的重点是什么？"
5. **得到回答** → AI从课件


In [7]:
# 先安装Hugging Face相关依赖（终端执行）
# pip install langchain-huggingface transformers torch

# 导入Hugging Face的LLM接口
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 1. 加载Hugging Face的模型和tokenizer
model_name = "./models/Qwen/Qwen3-0___6B"  # 模型名 这是本地路径 这里替换成自己的路径！！
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# 2. 构建pipeline
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=200,
    temperature=0.3
)

# 3. 初始化LangChain的LLM接口（统一接口）
hf_llm = HuggingFacePipeline(pipeline=pipe)

# 4. 调用模型（和之前的LLM调用方式完全一样）
prompt = "请用3句话解释什么是LangChain？"
result = hf_llm.invoke(prompt)

print("Hugging Face模型回复：")
print(result)

ModuleNotFoundError: No module named 'langchain_huggingface'

In [ ]:
#2.2提示词模板
from langchain_core.prompts import PromptTemplate

#1.定义提示词模板
#input_variables: 动态参数列表(这里是user_role和subject)
#template: 提示词模板字符串,用{参数名}表示动态参数
prompt_template=PromptTemplate(
    input_variables=["user_role","subject"],
    template="请给{user_role}写一段50字左右的{subject}学习建议,语言简洁实用,分2个小要点"
)

#2.格式化模板(传入具体参数,生成完整提示词)
# 给"高校学生"生成"LangChain"学习建议
formatted_promt=prompt_template.format(
    user_role="高校学生",
    subject="LangChain"
)
print("格式化后的提示词: ")
print(formatted_promt)



格式化后的提示词: 
请给高校学生写一段50字左右的LangChain学习建议,语言简洁实用,分2个小要点


In [ ]:
#3.调用模型生成结果
result=chat_model.invoke([{"role":"user","content":formatted_promt}])

print("\n生成的学习建议: ")
print(result.content)


生成的学习建议: 
**LangChain学习建议**

1. **先跑通再理解**：从官方Quickstart示例入手，动手搭建一个简单的对话链，边跑边看文档，建立直观感受。

2. **以项目驱动**：选一个小场景（如文档问答、知识库检索），拆解Chain→Agent→Memory的调用流程，在实践中掌握核心模块。


In [ ]:
#如果想给"程序员"生成"AI Agent"的学习建议,只需要修改format()中的参数
formatted_prompt=prompt_template.format(
    user_role="程序员",
    subject="AI Agent"
)
result=chat_model.invoke([{"role":"user","content":formatted_prompt}])
print("给程序员的AI Agent学习建议: ")
print(result.content)

给程序员的AI Agent学习建议: 
## AI Agent 学习建议

1. **掌握工具调用**：熟悉 Function Calling、API 编排，让 Agent 具备"手脚"。
2. **学好 Prompt 工程**：理解任务分解与状态管理，这是 Agent 的"大脑"。


In [ ]:
#2.2.2 少样本提示模板
"""
有时候，简单的提示词模板不足以让模型理解我们的需求——比如我们希望模型生成“特定格式”的内容（比如分点、带编号、有固定结构）。这时候就需要“少样本提示”：给模型看几个示例，让它照着示例的格式生成内容。
LangChain的FewShotPromptTemplate就是专门做这个的。
"""
#1.定义示例(少样本的核心:给模型看的参考案例)
examples=[
    {
        "subject":"Python编程",
        "method":"核心目标: 掌握基础语法和常用库；学习步骤：1. 学习变量、函数等基础语法 2. 实操小项目（如计算器） 3. 学习Pandas、Matplotlib库；注意事项：多动手实操，遇到错误及时调试。"
    },
    {
        "subject":"机器学习",
        "method":"核心目标：理解基础算法原理和应用场景；学习步骤：1. 复习数学基础（线性代数、概率） 2. 学习经典算法（线性回归、决策树） 3. 用Scikit-learn实操；注意事项：先理解原理，再动手实现，避免死记硬背。"
    }
]

In [ ]:
#2.定义示例模板(告诉框架如何解析示例)
example_template="""
学科:{subject}
学习方法: {method}
"""

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate,PromptTemplate

example_template=PromptTemplate(
    input_variables=["subject","method"],
    template=example_template
)

#3.定义最终的提示词模板(包含示例和用户需求)
few_shot_promt=FewShotPromptTemplate(
    examples=examples,              #传入示例
    example_prompt=example_template, #示例模板
    example_separator="\n",
    prefix="少样本提示词: ",
    suffix="参考以上样本, 回答: \n学科: {new_subject}\n学习方法: ",#最终给用户的提示(在示例之后
    input_variables=["new_subject"] #动态参数:用户要查询的新学科

)

c:\Users\DELL\AppData\Local\Programs\Python\Python310\lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [ ]:
#4.格式化模板(传入新学科:LangChain)
formatted_prompt=few_shot_promt.format(new_subject="LangChain")
print(formatted_prompt)

少样本提示词: 

学科:Python编程
学习方法: 核心目标: 掌握基础语法和常用库；学习步骤：1. 学习变量、函数等基础语法 2. 实操小项目（如计算器） 3. 学习Pandas、Matplotlib库；注意事项：多动手实操，遇到错误及时调试。


学科:机器学习
学习方法: 核心目标：理解基础算法原理和应用场景；学习步骤：1. 复习数学基础（线性代数、概率） 2. 学习经典算法（线性回归、决策树） 3. 用Scikit-learn实操；注意事项：先理解原理，再动手实现，避免死记硬背。

参考以上样本, 回答: 
学科: LangChain
学习方法: 


In [ ]:
#5.调用模型生成结果
result=chat_model.invoke([{"role":"user","content":formatted_prompt}])

print("生成的LangChain学习方法(AI回复)")
print(result.content)

生成的LangChain学习方法(AI回复)

